In [1]:
import sys
import numpy as np
import pybullet as p
import time
import logging
from typing import List, Tuple, Optional, Sequence, Collection, Dict, Any, cast
import random
import json

from predicators.structs import Action, Array, GroundAtom, Object, State, Type, ParameterizedOption
from predicators import utils
from predicators.settings import CFG
from gym.spaces import Box

#Import core environment methods, robot function etc.

from predicators.envs.pybullet_blocks import PyBulletBlocksEnv
from predicators.envs.pybullet_env import PyBulletEnv, create_pybullet_block
from predicators.pybullet_helpers.robots import SingleArmPyBulletRobot
from predicators.pybullet_helpers.robots.mobile_single_arm import MobileSingleArmPyBulletRobot
from predicators.pybullet_helpers.geometry import Pose
from predicators.pybullet_helpers.joint import JointPositions, get_joint_infos, get_joint_positions
from predicators.pybullet_helpers.link import get_link_state

#Import the functions that are to be tested:

from predicators.pybullet_helpers.motion_planning import run_motion_planning, run_base_motion_planning,\
                                                            run_coordinated_motion_planning
#The pick/place options to be tested are accessed via the env instance
from predicators.pybullet_helpers.controllers import execute_coordinated_path, create_move_end_effector_to_pose_option,\
                                                    create_change_fingers_option, create_move_base_option

pybullet build time: Jan 29 2025 23:16:28


In [2]:
logging.basicConfig(
    level=logging.WARNING,                    
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

#Defining test configuration, and overriding some default ones:
CFG.pybullet_robot = "fetch_mobile"
CFG.use_gui = True
#Draws helpful debug lines in the workspace.
#NOT SURE WHETHER TO USE THIS. WILL DECIDE AFTER A COUPLE RUNS.
#CFG.pybullet_draw_debug = True
#Initializing with standard size of blocks.
CFG.blocks_block_size = 0.05
CFG.pybullet_birrt_num_iters = 50
CFG.pybullet_birrt_num_attempts = 10
CFG.pybullet_birrt_smooth_amt = 20
CFG.seed = random.randint(0,10000)
#CFG.seed = 12
#Num of PyBullet physics steps per high-level Action in visualize_action_sequence
CFG.pybullet_sim_steps_per_action = 200

In [3]:
#Function to reset robot to a known pose
def reset_robot_fetch_mobile(robot: MobileSingleArmPyBulletRobot,
                             physics_client_id:int,
                             base_pose: Tuple[float, float, float] = (1.35, 0.75, 0.0), # (x,y,theta)
                             arm_joint_angle: Optional[List[float]]=None):
    """
    Resets the robot's base/ teleports it and arm to specified poses.

    """

    robot.move_base_to(base_pose, physics_client_id)
    if arm_joint_angle:
        #Set arm joints only
        robot.set_joints(arm_joint_angle)
    else:
        #robot.initial_joint_positions includes arm and finger joints
        robot.set_joints(robot.initial_joint_positions)
    #Step simulation a bit to allow PyBullet to settle the state.
    for _ in range(10):
        p.stepSimulation(physicsClientId=physics_client_id)


#Fn to create blocks in the env.
def create_test_block(env: PyBulletEnv,
                      pose: Tuple[float, float, float],
                      color: Tuple[float, float, float, float] = (0.8, 0.2, 0.2, 1.0),
                      name_suffix: str = "test") -> int:
    """
    Creates a single block at a specified pose for testing and returns its PyBullet ID.
    """

    #Use the fn defined in utils to create block
    block_id = create_pybullet_block(
        color,
        (CFG.blocks_block_size/2,)*3,
        env._obj_mass,
        env._obj_friction,
        env._default_orn,
        env._physics_client_id
    )
    #Place the block at desired pose.
    p.resetBasePositionAndOrientation(block_id, pose, env._default_orn, physicsClientId=env._physics_client_id)

    return block_id


#Get the list of all bodies except the robot.
#TODO: Need to add logic that saves object/body name
#      which can be used for better debugging with collision.
def get_all_non_robot_bodies(robot_id: int, physics_client_id:int) -> List[int]:
    """
    Gets all PyBullet body IDs in the simulation except for the robot itself.
    These are typically used as collision obstacles.
    """
    all_bodies = [p.getBodyUniqueId(i, physicsClientId=physics_client_id)
                    for i in range(p.getNumBodies(physicsClientId=physics_client_id))]


    return [b for b in all_bodies if b!=robot_id]


In [4]:
env = PyBulletBlocksEnv(use_gui=CFG.use_gui)

# Resets the environment to a specific task, getting an initial symbolic state.
# While initial_state_from_env is fetched, the option tests will create their own
# more specific symbolic states.
initial_state = env.reset("train", 0)

# The robot instance from the environment
robot = env._pybullet_robot
# The PyBullet physics client ID
physics_client_id = env._physics_client_id

assert isinstance(robot, MobileSingleArmPyBulletRobot), f"{robot} not an instance of {MobileSingleArmPyBulletRobot}."

# dyn = p.getDynamicsInfo(robot.robot_id, -1, physicsClientId=env._physics_client_id)
# print(f"\nMass, inertialFrame…{dyn}")
# input()


logging.info(f"Using robot: {robot.get_name()}")


#Store the robot's default arm and finger joint positions.
home_arm_joints = robot.initial_joint_positions

robot_obj = initial_state.get_objects(env._robot_type)[0]

#Store permament, fixed bodies
static_collision_bodies = get_all_non_robot_bodies(robot.robot_id, physics_client_id)

#print(f"Static_collision_bodies:{static_collision_bodies}")

#sys.exit(0)

#Define a rectangular workspace for base motion planning tests.
#(min_x, min_y, max_x, max_y)
workspace_bounds = (1.0, 0.2, 1.7, 1.3)

mark_pos = (1.5, 0.75, CFG.blocks_block_size / 2 + env.table_height+0.05)

p.addUserDebugText(
        "*",                          
        mark_pos,                     
        textColorRGB=[0, 0, 0],       
        textSize=1,                 
        lifeTime=0,                   
        physicsClientId=physics_client_id)

/home/cloaked04/anaconda3/envs/predicators/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


0

In [5]:
logging.info("\n--- Test 4: Coordinated Motion (Base + Arm) ---")
# Start further away and rotated, likely needing base movement
initial_base_pose_4 = (0.4, 0.6, -np.pi/2) 
reset_robot_fetch_mobile(robot, physics_client_id, base_pose=initial_base_pose_4, arm_joint_angle=home_arm_joints)
# print(f"[DEBUG] Home arm joints:{home_arm_joints}.")
# print(f"[DEBUG] Robot's joints after reset in test 4:{robot.get_joints()}.")
# print(f"[DEBUG] Symbolic state joints (STALE):    {env._current_state.simulator_state}")
#sys.exit(0)
home_orn = env.get_robot_ee_home_orn()
# Define a target EE pose that's likely out of reach for arm-only from initial_base_pose_4.
# A point on the table 1.35, 0.6, 0.2
z = env.table_height + CFG.blocks_block_size/2 + 0.05
#logging.critical(f"Value of z: {z}.")
orn = (0, 0.7071, 0, 0.7071)

target_ee_pose_4 = Pose(position=(1.5 , 0.75, z), 
                         orientation=home_orn)

# Create a block to be picked.
block_to_pick_pose_world = (1.5, 0.75, CFG.blocks_block_size / 2 + env.table_height)
logging.critical(f"World coords of block to pick:{block_to_pick_pose_world}.")

block_to_pick_id = create_test_block(env, pose=block_to_pick_pose_world, name_suffix="pick_target")

# Make a unique name for the symbolic object
symbolic_block_name = f"block{block_to_pick_id}" 
block_to_pick_obj_sym = Object(symbolic_block_name, env._block_type)

# Step 1: Update the environment's physical-to-symbolic map.
# This tells _get_state() that the new physical block ID now corresponds
# to a new symbolic object.
env._block_id_to_block[block_to_pick_id] = block_to_pick_obj_sym

#logging.debug(f"\nBlocks in the env:{env._block_id_to_block}.")

# Step 2: Get a handle to the official state object, which is mutable.
state_obj_to_modify = env._current_observation
#logging.debug(f"\nCurrent environment: {state_obj_to_modify}.")
assert isinstance(state_obj_to_modify, utils.PyBulletState), \
    f"Expected env._current_observation to be a PyBulletState, got {type(state_obj_to_modify)}"

state_obj_to_modify.data[block_to_pick_obj_sym] = np.zeros(len(env._block_type.feature_names))
fresh_state = env._get_state()
state_obj_to_modify.data = fresh_state.data
state_obj_to_modify.simulator_state = fresh_state.simulator_state 
state_obj_to_modify.base_pose = fresh_state.base_pose

static_collision_bodies = get_all_non_robot_bodies(robot.robot_id, physics_client_id)

coord_path_4_result = run_coordinated_motion_planning(
        robot,
        target_ee_pose=target_ee_pose_4,
        collision_bodies=static_collision_bodies,
        seed=CFG.seed,
        physics_client_id=physics_client_id,
        try_arm_only_first=True # Planner will try arm-only, fail, then try base+arm
    )

2025-07-24 21:18:36 root [CRITICAL] World coords of block to pick:(1.5, 0.75, 0.225).
2025-07-24 21:18:37 root [WARNING] Max time reached. No IKFast solution found.
2025-07-24 21:18:37 root [WARNING] No IK solutions found in 0.502 seconds
2025-07-24 21:18:37 predicators.pybullet_helpers.motion_planning [WARNING] 
 Collsion check:COLLIDES :-(


Coordinated Planning: Arm-only IK failed. Moving to base planning.


2025-07-24 21:18:38 root [WARNING] Max time reached. No IKFast solution found.
2025-07-24 21:18:38 root [WARNING] No IK solutions found in 0.501 seconds
2025-07-24 21:18:38 root [WARNING] Max time reached. No IKFast solution found.
2025-07-24 21:18:38 root [WARNING] No IK solutions found in 0.501 seconds
2025-07-24 21:18:38 predicators.pybullet_helpers.motion_planning [WARNING] Planning arm motion to above target.
2025-07-24 21:18:38 predicators.pybullet_helpers.motion_planning [WARNING] Length of path to above target: 21.
2025-07-24 21:18:38 predicators.pybullet_helpers.motion_planning [WARNING] IK to move down succeeeded.
2025-07-24 21:18:38 predicators.pybullet_helpers.motion_planning [WARNING] Planning to move EE down to target.
2025-07-24 21:18:38 predicators.pybullet_helpers.motion_planning [WARNING] Length of path to move down to target: 61.


In [6]:
current_state = env._get_state()
print(current_state)

PyBulletState(data={robby:robot: array([ 0.4       , -0.00359999,  0.54859996,  1.        ], dtype=float32), block0:block: array([1.3644842 , 1.0174147 , 0.22498886, 0.        , 0.51010406,
       0.3571398 , 0.3149855 ], dtype=float32), block1:block: array([1.3644813 , 1.0174148 , 0.27490628, 0.        , 0.3532403 ,
       0.9307069 , 0.95968556], dtype=float32), block2:block: array([1.3425561 , 0.49975476, 0.2249896 , 0.        , 0.7039258 ,
       0.9422027 , 0.24423918], dtype=float32), block9:block: array([1.5  , 0.75 , 0.225, 0.   , 0.8  , 0.2  , 0.2  ], dtype=float32)}, simulator_state=[-0.5661423031618853, 0.23765869714330612, -1.2426888976117905, -1.58572488798466, -1.816506151616821, -1.8930169282075866, 0.9399299011571235, 0.04, 0.04], base_pose=(0.4, 0.6, -1.5707963267948963))


In [7]:
base_path, arm_path = coord_path_4_result

In [8]:
robot.move_base_to(base_path[-1], physics_client_id)

In [9]:
robot.set_joints(arm_path[-1])

In [10]:
robot.get_state()[:3]

array([1.5       , 0.75000006, 0.27500007], dtype=float32)

In [ ]:
if coord_path_4_result:
        base_path_4, arm_path_4 = coord_path_4_result
        logging.info(f"Coordinated path (base+arm) found: Base waypoints: {len(base_path_4)}; Arm waypoints:{len(arm_path_4)}.")
        actions_4 = execute_coordinated_path(robot, base_path_4, arm_path_4, physics_client_id)
        # print(f"\n Last base position in base path: {base_path_4[-1]}")
        # print(f"\n Last full length action in the path: {actions_4[-1].arr}.")
        # print(f"\n Number of arm joints: {len(robot.arm_joints)}.")
        # print(f"\n Length of joint solution in action.arr: {len(actions_4[-1].arr)}")
        current_joint_positions = robot.get_joints()
        robot.move_base_to(base_path_4[-1], physics_client_id=physics_client_id)
        robot.set_joints(actions_4[-1].arr)
        ee_position_from_solution = robot.get_state()[:3]
        print(f"\nFinal EE position from last action returned by controller: {ee_position_from_solution}.")
        robot.move_base_to(initial_base_pose_4, physics_client_id=physics_client_id)
        robot.set_joints(current_joint_positions)
        input("Press Enter to proceed...")
        visualize_action_sequence(actions_4, "Test 4 Coordinated (Base+Arm)")
        time.sleep(0.1)

#final_ee_cache.append(new_ee)
else:
    logging.warning("Coordinated motion base planning failed for Test 4.")

print(f"Target EE pose: {target_ee_pose_4}.")